# 03 - Models and evaluation

**Stage 3 of 4.** Reads the modelling table from notebook 02, compares a naive baseline, logistic regression, k-NN and gradient boosting, and saves all results to `../results/`.

Rules followed (instructor's note, 1 Nov 2026): features only use earlier matches (checked in notebook 02) and every split is made by date, never at random.

Run from the folder `code/` in a fresh kernel. Shared functions are in `helpers.py`.

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

import helpers as h

DATA = Path("../data/processed/matches_model.csv.gz")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

table = pd.read_csv(DATA)
y_all = table["upset"].to_numpy()
print(f"{len(table):,} matches, {table['year'].min()}-{table['year'].max()}, upset rate {y_all.mean():.1%}")

70,564 matches, 2000-2024, upset rate 34.1%


## 1. Split by date

* **Development period:** 2000-2021. Hyperparameters are chosen here with an expanding-window cross-validation: for each validation year from 2015 to 2021 the models train on every earlier year and are scored on that year.
* **Test period:** 2022-2024, used once at the end. The final models train on 2000-2021 and are never refitted with test data.

A random split would let the model learn from matches played after the ones it predicts, which is exactly what the instructor's note rules out.

In [2]:
TEST_YEARS = (2022, 2024)
CV_YEARS = (2015, 2021)

dev_mask = (table["year"] < TEST_YEARS[0]).to_numpy()
test_mask = (table["year"] >= TEST_YEARS[0]).to_numpy()
dev = table[dev_mask].reset_index(drop=True)
test = table[test_mask].reset_index(drop=True)

folds = list(h.year_splits(dev, *CV_YEARS))
split_table = pd.DataFrame(
    [(y, len(tr), len(va), dev["upset"].to_numpy()[va].mean()) for y, tr, va in folds],
    columns=["validation year", "train matches", "validation matches", "upset rate"])
print(split_table.round(3).to_string(index=False))
print(f"\ntest 2022-2024: {len(test):,} matches, upset rate {test['upset'].mean():.1%}")

 validation year  train matches  validation matches  upset rate
            2015          44237                2767       0.308
            2016          47004                2769       0.319
            2017          49773                2748       0.340
            2018          52521                2858       0.358
            2019          55379                2601       0.387
            2020          57980                1387       0.357
            2021          59367                2622       0.363

test 2022-2024: 8,575 matches, upset rate 35.8%


## 2. Naive baseline

"The favourite always wins" predicts no upset in every match. Its accuracy is `1 - upset rate`, its AUC is 0.5 by construction and it never finds an upset (recall 0). Any model has to beat this accuracy and, more usefully, rank upsets better than chance.

In [3]:
base_val = np.concatenate([dev["upset"].to_numpy()[va] for _, _, va in folds])
print(f"baseline accuracy, validation years {CV_YEARS[0]}-{CV_YEARS[1]}: {1 - base_val.mean():.1%}")
print(f"baseline accuracy, test years {TEST_YEARS[0]}-{TEST_YEARS[1]}: {1 - test['upset'].mean():.1%}")

baseline accuracy, validation years 2015-2021: 65.4%
baseline accuracy, test years 2022-2024: 64.2%


## 3. Cross-validation over feature sets and models

Five feature sets answer the research question step by step:

* `rank`: the ranking gap only (log rank ratio, log points ratio);
* `rank+context`: adds surface, round, tournament level and best-of-3/5;
* `rank+elo`: the ranking gap plus the overall and surface-specific Elo difference;
* `all`: adds days since last match, recent form, surface record and head-to-head to `rank+context`;
* `all+elo`: `all` plus the two Elo differences.

The Elo ratings are an extension of the proposal (it listed Elo as an optional enrichment). They are computed in notebook 02 in the same chronological pass as the other features.

Missing values (a player with no earlier match in the data) are imputed with the training median inside each fold, together with a missing-value flag. Scaling is also fitted on the training part of each fold only. Gradient boosting handles missing values natively.

The configuration of each model and feature set with the lowest mean log loss over the seven validation years is selected. Log loss is used because it scores the probabilities, not only the 0/1 decisions.

In [4]:
GRIDS = {
    "logreg": [{"C": c} for c in (0.01, 0.1, 1.0, 10.0)],
    "knn": [{"k": k} for k in (50, 100, 200, 400)],
    "gboost": [{"depth": d, "n": n, "lr": lr}
               for d in (2, 3, 4) for n in (100, 300) for lr in (0.03, 0.1)],
}

rows, oof = [], {}
t0 = time.time()
for set_name, (num, cat) in h.FEATURE_SETS.items():
    cols = num + cat
    for kind, grid in GRIDS.items():
        for params in grid:
            preds = []
            for year, tr, va in folds:
                model = h.make_model(kind, num=num, cat=cat, **params)
                model.fit(dev.loc[tr, cols], dev["upset"].to_numpy()[tr])
                p = model.predict_proba(dev.loc[va, cols])[:, 1]
                s = h.scores(dev["upset"].to_numpy()[va], p, 0.5)
                rows.append({"features": set_name, "model": kind, "params": str(params),
                             "year": year, **s})
                preds.append(p)
            oof[(set_name, kind, str(params))] = np.concatenate(preds)
print(f"done in {time.time() - t0:.0f} s")
cv = pd.DataFrame(rows)
cv.to_csv(RESULTS / "cv_by_year.csv", index=False)

done in 307 s


In [5]:
summary = (cv.groupby(["features", "model", "params"])[["auc", "log_loss", "brier"]].mean()
             .reset_index())
best = summary.loc[summary.groupby(["features", "model"])["log_loss"].idxmin()]
best = best.sort_values(["features", "model"]).reset_index(drop=True)
best.to_csv(RESULTS / "cv_best_configs.csv", index=False)
best.round(4)

,features,model,params,auc,log_loss,brier
0,all,gboost,"{'depth': 3, 'n': 100, 'lr': 0.1}",0.6726,0.6002,0.2077
1,all,knn,{'k': 400},0.6667,0.6039,0.2092
2,all,logreg,{'C': 1.0},0.6694,0.6021,0.2084
3,all+elo,gboost,"{'depth': 3, 'n': 300, 'lr': 0.03}",0.6822,0.5967,0.2061
4,all+elo,knn,{'k': 400},0.6756,0.5993,0.2073
5,all+elo,logreg,{'C': 0.01},0.6802,0.5984,0.2067
6,rank,gboost,"{'depth': 3, 'n': 300, 'lr': 0.03}",0.6396,0.6163,0.2143
7,rank,knn,{'k': 400},0.6369,0.6172,0.2147
8,rank,logreg,{'C': 10.0},0.6395,0.6163,0.2142
9,rank+context,gboost,"{'depth': 4, 'n': 300, 'lr': 0.03}",0.6412,0.6144,0.2138


## 4. Decision threshold

Upsets are the minority class, so a threshold of 0.5 would almost never predict one. For each selected model the threshold is the one that maximises F1 for the upset class on the pooled out-of-fold predictions of the validation years. It is fixed before the test period is touched.

In [6]:
thr = {}
for _, r in best.iterrows():
    key = (r["features"], r["model"], r["params"])
    thr[(r["features"], r["model"])] = h.best_f1_threshold(base_val, oof[key])
pd.Series({f"{k[0]} / {k[1]}": round(v, 2) for k, v in thr.items()}, name="threshold")

all / gboost             0.26
all / knn                0.29
all / logreg             0.26
all+elo / gboost         0.27
all+elo / knn            0.28
all+elo / logreg         0.28
rank / gboost            0.25
rank / knn               0.25
rank / logreg            0.25
rank+context / gboost    0.24
rank+context / knn       0.27
rank+context / logreg    0.27
rank+elo / gboost        0.25
rank+elo / knn           0.26
rank+elo / logreg        0.27
Name: threshold, dtype: float64

## 5. Final fit and test evaluation (2022-2024)

Each selected model is refitted on all development years (2000-2021) and applied once to the test period.

In [7]:
y_test = test["upset"].to_numpy()
y_dev = dev["upset"].to_numpy()
test_rows, test_pred, fitted = [], {}, {}
for _, r in best.iterrows():
    num, cat = h.FEATURE_SETS[r["features"]]
    cols = num + cat
    params = eval(r["params"])
    model = h.make_model(r["model"], num=num, cat=cat, **params).fit(dev[cols], y_dev)
    p = model.predict_proba(test[cols])[:, 1]
    key = (r["features"], r["model"])
    fitted[key], test_pred[key] = model, p
    test_rows.append({"features": r["features"], "model": r["model"],
                      "threshold": thr[key], **h.scores(y_test, p, thr[key])})

# naive baseline on the same test set
test_rows.append({"features": "-", "model": "baseline (no upset)", "threshold": np.nan,
                  **h.scores(y_test, np.zeros(len(y_test)) + 1e-9, 0.5)})
test_metrics = pd.DataFrame(test_rows)
test_metrics.loc[test_metrics["model"].str.startswith("baseline"), ["auc", "log_loss", "brier"]] = [0.5, np.nan, np.nan]
test_metrics.to_csv(RESULTS / "test_metrics.csv", index=False)
test_metrics.round(3)

,features,model,threshold,auc,log_loss,brier,accuracy,precision,recall,f1
0,all,gboost,0.26,0.662,0.612,0.213,0.529,0.423,0.871,0.569
1,all,knn,0.29,0.657,0.614,0.214,0.537,0.426,0.845,0.567
2,all,logreg,0.26,0.657,0.614,0.214,0.528,0.422,0.861,0.566
3,all+elo,gboost,0.27,0.667,0.610,0.212,0.555,0.436,0.831,0.572
4,all+elo,knn,0.28,0.663,0.611,0.212,0.546,0.431,0.845,0.571
5,all+elo,logreg,0.28,0.666,0.611,0.213,0.564,0.440,0.802,0.569
6,rank,gboost,0.25,0.627,0.628,0.219,0.499,0.408,0.882,0.558
7,rank,knn,0.25,0.622,0.630,0.220,0.509,0.412,0.866,0.558
8,rank,logreg,0.25,0.627,0.628,0.219,0.500,0.408,0.883,0.559
9,rank+context,gboost,0.24,0.629,0.626,0.219,0.494,0.406,0.891,0.558


Accuracy at the tuned threshold is not comparable with the baseline accuracy: a model that flags many upsets gives up accuracy on the favourites. The comparison that matters is AUC, log loss and the precision/recall of the upset class.

In [8]:
pred_out = test[["tourney_id", "tourney_date", "year", "fav_id", "und_id", "upset"]].copy()
for (fs, kind), p in test_pred.items():
    pred_out[f"p_{fs}_{kind}"] = p.round(5)
pred_out.to_csv(RESULTS / "test_predictions.csv.gz", index=False)
print(pred_out.shape)

(8575, 21)


## 6. Does a non-linear model help, and does Elo add anything?

Three comparisons, each on the validation years and on the test period (bootstrap over test matches, 1,000 resamples, for the AUC difference):

1. **Non-linear against linear:** gradient boosting against logistic regression, both on `all+elo`.
2. **Elo beyond the ranking:** logistic regression on `rank+elo` against `rank`.
3. **Elo on top of everything else:** logistic regression on `all+elo` against `all`.

In [9]:
def selected(features, model):
    r = best[(best.features == features) & (best.model == model)].iloc[0]
    return r["params"]

def auc_by_year(features, model):
    d = cv[(cv.features == features) & (cv.model == model) & (cv.params == selected(features, model))]
    return d.set_index("year")["auc"]

cmp_years = pd.DataFrame({
    "rank logreg": auc_by_year("rank", "logreg"),
    "rank+elo logreg": auc_by_year("rank+elo", "logreg"),
    "all logreg": auc_by_year("all", "logreg"),
    "all+elo logreg": auc_by_year("all+elo", "logreg"),
    "all+elo gboost": auc_by_year("all+elo", "gboost"),
})
cmp_years["gboost - logreg (all+elo)"] = cmp_years["all+elo gboost"] - cmp_years["all+elo logreg"]
cmp_years["Elo effect, rank"] = cmp_years["rank+elo logreg"] - cmp_years["rank logreg"]
cmp_years["Elo effect, all"] = cmp_years["all+elo logreg"] - cmp_years["all logreg"]
cmp_years.loc["mean"] = cmp_years.mean()
cmp_years.round(4).to_csv(RESULTS / "auc_by_validation_year.csv")
cmp_years.round(4)

,rank logreg,rank+elo logreg,all logreg,all+elo logreg,all+elo gboost,gboost - logreg (all+elo),"Elo effect, rank","Elo effect, all"
year,,,,,,,,
2015,0.6729,0.7047,0.6950,0.7045,0.7045,0.0000,0.0318,0.0096
2016,0.6455,0.6925,0.6782,0.6952,0.7005,0.0053,0.0470,0.0171
2017,0.6343,0.6863,0.6721,0.6886,0.6906,0.0019,0.0521,0.0165
2018,0.6204,0.6569,0.6478,0.6630,0.6656,0.0027,0.0365,0.0152
2019,0.6361,0.6770,0.6674,0.6785,0.6806,0.0020,0.0410,0.0111
2020,0.6279,0.6636,0.6614,0.6690,0.6702,0.0012,0.0356,0.0076
2021,0.6397,0.6603,0.6643,0.6628,0.6630,0.0002,0.0206,-0.0015
mean,0.6395,0.6773,0.6694,0.6802,0.6822,0.0019,0.0378,0.0108


In [10]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
P = {k: test_pred[k] for k in test_pred}
pairs = {
    "AUC gboost - logreg (all+elo)": (("all+elo", "gboost"), ("all+elo", "logreg")),
    "AUC Elo effect on rank only (logreg)": (("rank+elo", "logreg"), ("rank", "logreg")),
    "AUC Elo effect on all features (logreg)": (("all+elo", "logreg"), ("all", "logreg")),
    "AUC all+elo gboost - rank logreg": (("all+elo", "gboost"), ("rank", "logreg")),
}
n = len(y_test)
diffs = {k: [] for k in pairs}
for _ in range(1000):
    i = rng.integers(0, n, n)
    for name, (a, b) in pairs.items():
        diffs[name].append(roc_auc_score(y_test[i], P[a][i]) - roc_auc_score(y_test[i], P[b][i]))
def ci(d):
    return f"{np.mean(d):+.4f}  95% CI [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]"
boot = pd.Series({k: ci(v) for k, v in diffs.items()})
boot.to_csv(RESULTS / "test_bootstrap.csv", header=["difference"])
print(boot.to_string())

AUC gboost - logreg (all+elo)              +0.0015  95% CI [-0.0012, +0.0041]
AUC Elo effect on rank only (logreg)       +0.0362  95% CI [+0.0269, +0.0455]
AUC Elo effect on all features (logreg)    +0.0084  95% CI [+0.0033, +0.0131]
AUC all+elo gboost - rank logreg           +0.0399  95% CI [+0.0302, +0.0494]


## 7. Which features matter

Permutation importance on the test period for the selected gradient boosting model on `all+elo`: the columns of one feature group are shuffled together (5 repetitions) and the drop in AUC is recorded. A large drop means the model relies on that group.

In [11]:
model = fitted[("all+elo", "gboost")]
num, cat = h.FEATURE_SETS["all+elo"]
cols = num + cat
base_auc = roc_auc_score(y_test, model.predict_proba(test[cols])[:, 1])
rng = np.random.default_rng(1)
imp_rows = []
for group, gcols in h.FEATURE_GROUPS.items():
    drops = []
    for _ in range(5):
        shuffled = test[cols].copy()
        order = rng.permutation(len(shuffled))
        shuffled[gcols] = shuffled[gcols].to_numpy()[order]
        drops.append(base_auc - roc_auc_score(y_test, model.predict_proba(shuffled)[:, 1]))
    imp_rows.append({"group": group, "auc_drop_mean": np.mean(drops), "auc_drop_std": np.std(drops)})
importance = pd.DataFrame(imp_rows).sort_values("auc_drop_mean", ascending=False)
importance.to_csv(RESULTS / "permutation_importance.csv", index=False)
print(f"test AUC of the model: {base_auc:.4f}")
importance.round(4)

test AUC of the model: 0.6674


,group,auc_drop_mean,auc_drop_std
6,Elo ratings,0.1036,0.0062
0,ranking gap,0.0230,0.0010
1,"surface, round, level, best of",0.0024,0.0011
3,recent form,0.0017,0.0002
2,days since last match,0.0017,0.0006
4,surface record,0.0011,0.0003
5,head-to-head,0.0005,0.0001


## 8. Calibration and year-by-year behaviour

Calibration compares the predicted upset probability with the observed upset rate in bins of predicted probability (test period). The year table shows whether the result holds in each test year.

In [12]:
cal_rows = []
for (fs, kind), p in test_pred.items():
    if fs != "all+elo":
        continue
    bins = pd.qcut(p, 10, duplicates="drop")
    g = pd.DataFrame({"p": p, "y": y_test, "bin": bins}).groupby("bin", observed=True)
    cal = g.agg(mean_predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size")).reset_index(drop=True)
    cal.insert(0, "model", kind)
    cal_rows.append(cal)
calibration = pd.concat(cal_rows)
calibration.to_csv(RESULTS / "calibration.csv", index=False)

year_rows = []
for yr in sorted(test["year"].unique()):
    m = (test["year"] == yr).to_numpy()
    for (fs, kind), p in test_pred.items():
        year_rows.append({"year": yr, "features": fs, "model": kind,
                          "auc": roc_auc_score(y_test[m], p[m])})
year_auc = pd.DataFrame(year_rows).pivot_table(index=["features", "model"], columns="year", values="auc")
year_auc.to_csv(RESULTS / "test_auc_by_year.csv")
print(calibration[calibration.model == "gboost"].round(3).to_string(index=False))
print()
print(year_auc.round(3).to_string())

 model  mean_predicted  observed   n
gboost           0.098     0.093 858
gboost           0.182     0.212 857
gboost           0.238     0.261 858
gboost           0.283     0.320 857
gboost           0.322     0.355 858
gboost           0.362     0.376 857
gboost           0.407     0.446 857
gboost           0.453     0.470 858
gboost           0.509     0.471 857
gboost           0.610     0.575 858

year                  2022   2023   2024
features     model                      
all          gboost  0.675  0.652  0.660
             knn     0.676  0.643  0.653
             logreg  0.676  0.643  0.654
all+elo      gboost  0.681  0.656  0.666
             knn     0.679  0.649  0.662
             logreg  0.685  0.651  0.663
rank         gboost  0.629  0.633  0.620
             knn     0.626  0.626  0.615
             logreg  0.630  0.633  0.620
rank+context gboost  0.632  0.632  0.622
             knn     0.625  0.629  0.623
             logreg  0.633  0.631  0.621
rank+elo     gboos

Everything printed above is also saved in `../results/` and is the input of notebook 04, which draws the figures.